# Multi-Tag Trend Plots — Configurable Period Viewer

Generates one interactive HTML plot per configurable time period (default: 6-month windows across 2024 and 2025).

Each plot shows:
- **Normalized PV / SP / OP trends** split into small capped files (Controllers, then Temperature/Flow/Level/Pressure indicators, then Other; big families auto-split) so each HTML loads fast
- **Control actions subplot** — OP / SP / MODE changes as scatter markers, colour-coded by direction

**Pipeline stages:**
1. Load PV/OP/SP time series from parquet
2. Load & preprocess CHANGE events (dedup → start-date cutoff → filter OP/SP/MODE)
3. Trip period filtering (both time series and events)
4. Build colour palette, ordered tag list, and split tags into groups
5. Generate one HTML + CSV per period × group → `RESULTS/new_rca_plots_<timestamp>/`

**Configuration**: Edit the config cell (Section 1) to change tag list, years, period length, or file paths.

## Section 1: Imports & Configuration

In [26]:
import os
import calendar
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit here to switch files, years, or period length
# ═══════════════════════════════════════════════════════════════════════════════

# ── Data files ──
PV_OP_FILE            = '/home/h604827/ControlActions/DATA/new_rca_pv_op_data/merged_all_historian_tags.parquet'
EVENTS_FILE           = ''   # ← Set path when events file is available; leave '' to skip
TRIP_FILE             = '/home/h604827/ControlActions/DATA/Final_List_Trip_Duration.csv'
OPERATING_LIMITS_FILE = '/home/h604827/ControlActions/DATA/operating_limits.xlsx'

# ── Preprocessing ──
FILTER_TRIPS = True
START_DATE   = '2024-01-01'   # Rows before this date are dropped from events

# ── Period configuration ──
YEARS         = [2024, 2025]   # Years to generate plots for
PERIOD_MONTHS = 1              # Period length: 6 = semi-annual (H1/H2), 3 = quarterly (Q1–Q4), 1 = monthly

# ── Plot downsampling (HUGE speed/size lever; raw CSVs stay full-resolution) ──
# 1-min data = ~44k rows/month/trace → multi-MB HTML. Resampling the *plot only*
# to e.g. 10 min keeps shapes intact but shrinks files ~10×. '' = no resample.
PLOT_RESAMPLE = '30min'

# ── Default-visible tag ──
FI1000_COL = '02FI_1000.PV'   # Only this trace is shown by default; all others are legend-only

# ── Tag grouping → one HTML per group per month (keeps each file small/fast) ──
# Tags are bucketed by instrument family, then split so no plot exceeds
# MAX_TAGS_PER_PLOT (smaller = faster to load). Buckets: controllers (xIC/HIC/SIC,
# PV+SP+OP), then Temperature/Flow/Level/Pressure indicators, then Other.
SPLIT_TAGS_INTO_GROUPS = True
MAX_TAGS_PER_PLOT      = 75   # cap per plot; only families bigger than this auto-split into ...1/...2

# ── Plot rendering ──
# Use CDN to load Plotly JS instead of embedding it in each file (~3 MB savings per file).
# Set to True if working offline.
PLOTLY_EMBED_JS = False

# ── Output directory (with IST run timestamp) ──
_run_ts_ist = pd.Timestamp.now(tz='Asia/Kolkata').strftime('%d%b%Y_%H%M').upper()
RESULTS_DIR = Path(f'/home/h604827/ControlActions/RESULTS/new_rca_plots_{_run_ts_ist}')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PV/OP file     : {PV_OP_FILE}")
print(f"Events file    : {EVENTS_FILE or '(not set — will skip)'}")
print(f"Trip filter    : {'ON' if FILTER_TRIPS else 'OFF'}")
print(f"Start date     : {START_DATE}")
print(f"Years          : {YEARS}  |  Period length: {PERIOD_MONTHS} months")
print(f"FI1000 col     : {FI1000_COL}")
print(f"Results dir    : {RESULTS_DIR}")

PV/OP file     : /home/h604827/ControlActions/DATA/new_rca_pv_op_data/merged_all_historian_tags.parquet
Events file    : (not set — will skip)
Trip filter    : ON
Start date     : 2024-01-01
Years          : [2024, 2025]  |  Period length: 1 months
FI1000 col     : 02FI_1000.PV
Results dir    : /home/h604827/ControlActions/RESULTS/new_rca_plots_29JUN2026_1337


## Section 2: Target Tags

Normalized from the provided list (`03-FIC-1085` → `03FIC_1085`).  
Cell 4 will intersect these against actual parquet columns and print which are missing.

In [2]:
# Normalized tag names (03-FIC-1085 → 03FIC_1085, instrument type preserved)
TARGET_TAGS = [
    # Flow controllers / indicators
    '03FIC_1085',
    '03FIC_1141',
    '03FIC_1151A',
    '03FIC_1474',
    '03FIC_2474',
    '03FIC_3227',
    '03FIC_3415',
    '03FIC_3435',
    '03FICA_1258',
    '03FICA_1668',
    '03FICA_2258',
    # Level controllers
    '03LIC_1603',
    '03LICA_1016',
    '03LICA_1035',
    '03LICA_1071',
    '03LICA_1094',
    '03LICA_1097',
    '03LICA_1172',
    '03LICA_1183',
    '03LICA_1608',
    '03LICA_1618',
    '03LICA_1619',
    '03LICA_3153',
    '03LICA_3178',
    '03LICA_3411',
    '03LICSA_1132',
    '03LICSA_1272',
    # Pressure controllers / indicators
    '03PIC_1023',
    '03PIC_1141A',
    '03PIC_1141B',
    '03PIC_1151B',
    '03PIC_1186',
    '03PIC_1620',
    '03PIC_1720',
    '03PIC_3002',
    '03PIC_3440',
    '03PICA_1013',
    '03PICA_1068',
    '03PICA_1104',
    '03PICA_1151C',
    '03PICA_3000',
    '03PICA_3131',
    '03PICA_3252',
    # Temperature controllers / indicators
    '03TIC_1023',
    '03TIC_1092',
    '03TIC_1142',
    '03TIC_1145',
    '03TIC_1302',
    '03TIC_1671',
    '03TIC_1745A',
    '03TIC_1745B',
    '03TIC_1892',
    '03TIC_2892',
    '03TIC_3008',
    '03TICA_1009',
    '03TICA_1635',
    '03TICA_3014',
    '03TICA_3148',
    # Plant-level indicators (non-controllable)
    '02FI_1000',    # Feed flow (FI1000 — reference signal, shown by default)
    '03TI_1005',
    '03TI_1421',
    '03TI_1405',
]

print(f"Total target tags defined: {len(TARGET_TAGS)}")


Total target tags defined: 62


## Section 3: Load PV / OP / SP Time Series

In [3]:
def strip_timezone(dt_series):
    """Remove timezone info from a datetime Series if present."""
    if getattr(dt_series.dt, 'tz', None) is not None:
        return dt_series.dt.tz_localize(None)
    return dt_series


# ── Load PV/OP/SP time series ──
op_pv_data_df = pd.read_parquet(PV_OP_FILE)

# Handle TimeStamp as column or as index
if 'TimeStamp' in op_pv_data_df.columns:
    op_pv_data_df['TimeStamp'] = pd.to_datetime(op_pv_data_df['TimeStamp'])
    op_pv_data_df['TimeStamp'] = strip_timezone(op_pv_data_df['TimeStamp'])
    op_pv_data_df.set_index('TimeStamp', inplace=True)
else:
    op_pv_data_df.index = strip_timezone(pd.to_datetime(op_pv_data_df.index))

op_pv_data_df.sort_index(inplace=True)

# ── Identify PV / OP / SP columns ──
all_data_cols = list(op_pv_data_df.columns)
pv_cols = sorted([c for c in all_data_cols if c.endswith('.PV')])
op_cols = sorted([c for c in all_data_cols if c.endswith('.OP')])
sp_cols = sorted([c for c in all_data_cols if c.endswith('.SP')])

print(f"Loaded  : {op_pv_data_df.shape[0]:,} rows  |  "
      f"{op_pv_data_df.index.min()} → {op_pv_data_df.index.max()}")
print(f"Columns : {len(pv_cols)} PV  |  {len(op_cols)} OP  |  {len(sp_cols)} SP")
print(f"Other columns: {[c for c in all_data_cols if not (c.endswith('.PV') or c.endswith('.OP') or c.endswith('.SP'))]}")

# ── Check TARGET_TAGS coverage ──
# Build the set of base tags that have any column in the parquet
present_bases = set(c.rsplit('.', 1)[0] for c in all_data_cols)
present_targets   = [t for t in TARGET_TAGS if t in present_bases]
missing_targets   = [t for t in TARGET_TAGS if t not in present_bases]

print(f"\nTarget tag coverage: {len(present_targets)}/{len(TARGET_TAGS)} found in parquet")
if missing_targets:
    print(f"  Missing tags: {missing_targets}")
else:
    print("  All target tags present!")

# ── Load operating limits ──
tag_operating_limits = {}
try:
    op_limits_raw = pd.read_excel(OPERATING_LIMITS_FILE)
    for _, row in op_limits_raw.iterrows():
        tag   = row['TagName']
        upper = row.get('NEW_UPPER_LIMIT', np.nan)
        lower = row.get('NEW_LOWER_LIMIT', np.nan)
        if pd.isna(upper) or (isinstance(upper, str) and 'NOT' in upper.upper()):
            upper = row.get('OLD_UPPER_LIMIT', np.nan)
        if pd.isna(lower) or (isinstance(lower, str) and 'NOT' in lower.upper()):
            lower = row.get('OLD_LOWER_LIMIT', np.nan)
        try:
            tag_operating_limits[tag] = {'upper': float(upper), 'lower': float(lower)}
        except (ValueError, TypeError):
            pass
    print(f"\nOperating limits loaded for {len(tag_operating_limits)} tags")
except Exception as e:
    print(f"\nOperating limits file not loaded: {e}")


Loaded  : 2,102,317 rows  |  2022-01-03 22:45:00 → 2026-01-09 23:29:00
Columns : 300 PV  |  60 OP  |  1 SP
Other columns: []

Target tag coverage: 18/62 found in parquet
  Missing tags: ['03FIC_1141', '03FIC_1151A', '03FIC_2474', '03FIC_3227', '03FICA_1258', '03FICA_1668', '03FICA_2258', '03LICA_1016', '03LICA_1035', '03LICA_1071', '03LICA_1094', '03LICA_1097', '03LICA_1172', '03LICA_1183', '03LICA_1608', '03LICA_1618', '03LICA_1619', '03LICA_3153', '03LICA_3178', '03LICA_3411', '03LICSA_1132', '03LICSA_1272', '03PIC_1141A', '03PIC_1141B', '03PIC_1151B', '03PIC_1186', '03PIC_1720', '03PIC_3440', '03PICA_1013', '03PICA_1068', '03PICA_1104', '03PICA_1151C', '03PICA_3000', '03PICA_3131', '03PICA_3252', '03TIC_1302', '03TIC_1745A', '03TIC_1745B', '03TIC_1892', '03TIC_2892', '03TICA_1009', '03TICA_1635', '03TICA_3014', '03TICA_3148']

Operating limits loaded for 56 tags


## Section 3b: Load ADNOC Tag Info & Map Assets → UUIDs → Events Folders

Loads `DATA/ADNOC_Tag Info.xlsx`, maps each tag's `AssetName` (e.g. `1F`) to a UUID via  
`DATA/config/config/EMDB/ADNOC-B.json`, then checks which UUID folders exist in  
`Historian_Events/events/` so we know which assets have events data available.

In [4]:
import json
from IPython.display import display

# ── Paths ──
ADNOC_TAG_INFO_FILE = '/home/h604827/ControlActions/DATA/ADNOC_Tag Info.xlsx'
ADNOC_JSON_FILE     = '/home/h604827/ControlActions/DATA/config/config/EMDB/ADNOC-B.json'
EVENTS_DIR          = '/home/h604827/ControlActions/DATA/Historian_Events/events'

# ── Load ADNOC Tag Info Excel (header on row index 1) ──
df_tag_info = pd.read_excel(ADNOC_TAG_INFO_FILE, header=1)
df_tag_info.columns = df_tag_info.columns.str.strip()

# Keep rows with a real DCSTag (not '-', NaN, or blanks)
df_tag_info = df_tag_info[
    df_tag_info['DCSTag'].notna() &
    ~df_tag_info['DCSTag'].astype(str).str.strip().isin(['-', '', 'nan'])
].copy().reset_index(drop=True)
df_tag_info['DCSTag']    = df_tag_info['DCSTag'].astype(str).str.strip()
df_tag_info['AssetName'] = df_tag_info['AssetName'].astype(str).str.strip()

print(f"ADNOC Tag Info loaded : {len(df_tag_info)} tags with valid DCSTag")

# ── Load ADNOC-B.json → AssetName → UUID map ──
with open(ADNOC_JSON_FILE, 'r') as _f:
    _asset_nodes = json.load(_f)
asset_name_to_uuid = {n['Name']: n['Id'] for n in _asset_nodes if n.get('LeafNode')}
print(f"ADNOC-B.json leaf nodes loaded : {len(asset_name_to_uuid)} entries")

# ── Check events folder availability ──
events_on_disk = set(os.listdir(EVENTS_DIR))
print(f"UUID folders in events/       : {len(events_on_disk)}")

# ── Build per-tag availability table ──
coverage_rows = []
for _, row in df_tag_info.iterrows():
    asset  = row['AssetName']
    uuid   = asset_name_to_uuid.get(asset)
    has_ev = uuid in events_on_disk if uuid else False
    coverage_rows.append({
        'Tags'       : row['Tags'],
        'DCSTag'     : row['DCSTag'],
        'AssetName'  : asset,
        'UUID'       : uuid or 'NOT IN JSON',
        'EventsData' : 'YES' if has_ev else 'NO',
    })

df_coverage = pd.DataFrame(coverage_rows)

n_with_events    = (df_coverage['EventsData'] == 'YES').sum()
n_without_events = (df_coverage['EventsData'] == 'NO').sum()

print(f"\nCoverage summary:")
print(f"  Tags with events data    : {n_with_events}")
print(f"  Tags without events data : {n_without_events}")
print(f"\nFull coverage table:")
display(df_coverage)


ADNOC Tag Info loaded : 57 tags with valid DCSTag
ADNOC-B.json leaf nodes loaded : 69 entries
UUID folders in events/       : 70

Coverage summary:
  Tags with events data    : 56
  Tags without events data : 1

Full coverage table:


,Tags,DCSTag,AssetName,UUID,EventsData
0,03-FIC-1085,03FIC_1085,1F,ccfff937-ff8c-4f84-a31c-a41f00fa0ff8,YES
1,03-FIC-1141,03FI_1141,1L,65f25414-8750-4713-b7a5-d073ee37c3d5,YES
2,03-FIC-1151A,03FI_1151,1L,65f25414-8750-4713-b7a5-d073ee37c3d5,YES
3,03-FIC-1474,03FIC_1474,3I,e0f2c384-4118-4662-99a4-4957e28ae60e,YES
4,03-FIC-2474,03FIC_2474,3I,e0f2c384-4118-4662-99a4-4957e28ae60e,YES
5,03-FIC-3227,03FIC_3227,1H,f84c869c-4c09-4485-9cdd-35a994fc2350,YES
6,03-FIC-3415,03FIC_3415,1K,6024200f-19c3-422a-a0e3-695a29031803,YES
7,03-FIC-3435,03FIC_3435,1K,6024200f-19c3-422a-a0e3-695a29031803,YES
8,03-FICA-1258,03FIC_1258,3I,e0f2c384-4118-4662-99a4-4957e28ae60e,YES
9,03-FICA-1668,03FIC_1668,1O,df0dd88a-ed6d-412b-9d36-6c35829939cd,YES


## Section 3c: Load Events from Historian Folders, Merge & Preprocess

For each unique asset UUID that has an `events/` folder:
- Loads the `_E.parquet` file (operator/process change events — contains OP/SP/MODE actions)
- Merges all assets into a single DataFrame
- Applies the timezone offset: events are stored in UTC+5:30 (IST); subtract 1.5 h to align with PV data (UAE local, UTC+4)
- Deduplicates on `[VT_Start, Source, ConditionName, Description]`
- Filters to `ConditionName = 'CHANGE'` and `Description` in `['OP', 'SP', 'MODE']`
- Sets the `change_events` variable used by all downstream cells

In [5]:
# ── Unique asset UUIDs that have an events folder ──
unique_assets_with_events = (
    df_coverage[df_coverage['EventsData'] == 'YES'][['AssetName', 'UUID']]
    .drop_duplicates(subset=['UUID'])
    .reset_index(drop=True)
)
print(f"Unique assets with events folder : {len(unique_assets_with_events)}")
print(unique_assets_with_events.to_string(index=False))

# ── Load _E.parquet for each available asset ──
# _E files hold process/operator change events (OP, SP, MODE actions)
TIME_OFFSET = pd.Timedelta(hours=1.5)   # UTC+5:30 (IST) → UAE local (UTC+4)

raw_parts = []
for _, arow in unique_assets_with_events.iterrows():
    asset_name = arow['AssetName']
    uuid       = arow['UUID']
    asset_dir  = os.path.join(EVENTS_DIR, uuid)

    try:
        files = os.listdir(asset_dir)
    except FileNotFoundError:
        print(f"  [SKIP] {asset_name} — folder not accessible")
        continue

    # Load _E file only (change events)
    e_files = [f for f in files if f.endswith('_E.parquet')]
    if not e_files:
        print(f"  [SKIP] {asset_name} — no _E.parquet found (files: {files})")
        continue

    fp = os.path.join(asset_dir, e_files[0])
    df_ev = pd.read_parquet(fp)

    # Parse VT_Start — use format='mixed' to handle mixed ms/no-ms rows
    if 'VT_Start' in df_ev.columns:
        df_ev['VT_Start'] = pd.to_datetime(df_ev['VT_Start'], format='mixed')
    df_ev['_asset_name'] = asset_name
    raw_parts.append(df_ev)
    print(f"  ✓ {asset_name:5s} ({uuid[:8]}...)  {e_files[0]:55s}  {len(df_ev):>10,} rows")

print(f"\nRaw event tables loaded : {len(raw_parts)} assets")

# ── Merge ──
df_raw_events = pd.concat(raw_parts, ignore_index=True, sort=False)
print(f"Merged shape            : {df_raw_events.shape}")
print(f"Columns                 : {df_raw_events.columns.tolist()}")

# ── Apply timezone offset ──
print(f"\nTimezone offset: VT_Start -= {TIME_OFFSET}  (IST → UAE local)")
print(f"  Before : {df_raw_events['VT_Start'].min()}  →  {df_raw_events['VT_Start'].max()}")
df_raw_events['VT_Start'] = df_raw_events['VT_Start'] - TIME_OFFSET
print(f"  After  : {df_raw_events['VT_Start'].min()}  →  {df_raw_events['VT_Start'].max()}")

# Strip any remaining timezone info so comparisons work with naive PV timestamps
if hasattr(df_raw_events['VT_Start'].dt, 'tz') and df_raw_events['VT_Start'].dt.tz is not None:
    df_raw_events['VT_Start'] = df_raw_events['VT_Start'].dt.tz_localize(None)

# ── Sort ──
df_raw_events.sort_values('VT_Start', inplace=True, ignore_index=True)

# ── Deduplication ──
dedup_cols = ['VT_Start', 'Source', 'ConditionName', 'Description']
# Only dedup on columns that actually exist
dedup_cols = [c for c in dedup_cols if c in df_raw_events.columns]
pre_dedup = len(df_raw_events)
df_raw_events = (
    df_raw_events
    .groupby(dedup_cols, sort=False)
    .first()
    .reset_index()
    .sort_values('VT_Start')
    .reset_index(drop=True)
)
print(f"\nDedup : {pre_dedup:,} → {len(df_raw_events):,}  (removed {pre_dedup - len(df_raw_events):,})")

# ── Start-date cutoff ──
df_raw_events = df_raw_events[
    df_raw_events['VT_Start'] >= pd.to_datetime(START_DATE)
].reset_index(drop=True)
print(f"After start-date cutoff ({START_DATE}): {len(df_raw_events):,} rows")

# ── Filter to OP / SP / MODE CHANGE events ──
change_events = df_raw_events[
    (df_raw_events['ConditionName'] == 'CHANGE') &
    (df_raw_events['Description'].isin(['OP', 'SP', 'MODE']))
][['Source', 'Description', 'VT_Start', 'PrevValue', 'Value']].copy().reset_index(drop=True)

# ── Compute action direction ──
_val  = pd.to_numeric(change_events['Value'],     errors='coerce')
_prev = pd.to_numeric(change_events['PrevValue'], errors='coerce')
change_events['action_direction'] = np.where(
    _val > _prev, 'increase',
    np.where(_val < _prev, 'decrease', 'no_change')
)

print(f"\n{'='*60}")
print(f"change_events (OP/SP/MODE) : {len(change_events):,} rows")
print(f"  By Description : {change_events['Description'].value_counts().to_dict()}")
print(f"  Unique sources : {change_events['Source'].nunique()}")
print(f"  Date range     : {change_events['VT_Start'].min()}  →  {change_events['VT_Start'].max()}")
print(f"{'='*60}")


Unique assets with events folder : 12
AssetName                                 UUID
       1F ccfff937-ff8c-4f84-a31c-a41f00fa0ff8
       1L 65f25414-8750-4713-b7a5-d073ee37c3d5
       3I e0f2c384-4118-4662-99a4-4957e28ae60e
       1H f84c869c-4c09-4485-9cdd-35a994fc2350
       1K 6024200f-19c3-422a-a0e3-695a29031803
       1O df0dd88a-ed6d-412b-9d36-6c35829939cd
       1E 072404dc-ae93-4239-9f6b-f4624391c391
       1D 9cac06f4-131b-4af0-aea0-17b223973d12
       1G 1eb386f1-e52a-4f80-a5e6-de5475441be7
       1J 49fe1f01-8f3d-4383-9a36-0d77de120771
       1I 0b4d6b2d-7ed9-42ad-8ed6-6ee2bd33172c
       1N be29d6b0-3795-4aee-a022-cf392a1efc18
  ✓ 1F    (ccfff937...)  2021011814_2026011001_E.parquet                             138,249 rows
  ✓ 1L    (65f25414...)  2021011814_2026011001_E.parquet                           1,051,748 rows
  ✓ 3I    (e0f2c384...)  2021011814_2026011001_E.parquet                             388,599 rows
  ✓ 1H    (f84c869c...)  2021011814_2026011001_E.parquet 

## Section 4: Load & Preprocess Events Data

If `EVENTS_FILE` is empty the cell prints a warning and sets `change_events = None` — all subsequent steps handle this gracefully (no control-actions subplot will be generated).

In [6]:
# ── FALLBACK: only runs if Section 3c did not already set change_events ──
# Use this if you want to load events from a pre-built CSV/parquet instead of
# the UUID-based historian folders (e.g. trip_filtered_events_dedup.csv).
if 'change_events' in dir() and change_events is not None:
    print(f"change_events already loaded by Section 3c  →  {len(change_events):,} rows")
    print("Section 4 fallback skipped.")
elif not EVENTS_FILE:
    print("WARNING: EVENTS_FILE is not set and Section 3c produced no events.")
    print("  Control-actions subplot will be omitted from all plots.")
    change_events = None
else:
    # ── Load from flat file ──
    _path = EVENTS_FILE
    if _path.endswith('.parquet'):
        events_df = pd.read_parquet(_path)
    else:
        events_df = pd.read_csv(_path, low_memory=False)

    events_df['VT_Start'] = pd.to_datetime(events_df['VT_Start'])
    events_df['VT_Start'] = strip_timezone(events_df['VT_Start'])
    events_df = events_df.sort_values('VT_Start').reset_index(drop=True)
    print(f"Loaded  : {len(events_df):,} rows  |  "
          f"{events_df['VT_Start'].min()} → {events_df['VT_Start'].max()}")

    dedup_cols = ['VT_Start', 'Source', 'ConditionName', 'Description']
    pre_dedup  = len(events_df)
    events_df  = events_df.groupby(dedup_cols, sort=False).first().reset_index()
    events_df  = events_df.sort_values('VT_Start').reset_index(drop=True)
    print(f"Dedup   : {pre_dedup:,} → {len(events_df):,}  (removed {pre_dedup - len(events_df):,})")

    events_df = events_df[events_df['VT_Start'] >= pd.to_datetime(START_DATE)].reset_index(drop=True)
    print(f"After start-date cutoff ({START_DATE}): {len(events_df):,} rows")

    change_events = events_df[
        (events_df['ConditionName'] == 'CHANGE') &
        (events_df['Description'].isin(['OP', 'SP', 'MODE']))
    ][['Source', 'Description', 'VT_Start', 'PrevValue', 'Value']].copy().reset_index(drop=True)

    _val  = pd.to_numeric(change_events['Value'],     errors='coerce')
    _prev = pd.to_numeric(change_events['PrevValue'], errors='coerce')
    change_events['action_direction'] = np.where(
        _val > _prev, 'increase', np.where(_val < _prev, 'decrease', 'no_change')
    )

    print(f"\nCHANGE events (OP/SP/MODE): {len(change_events):,} rows")
    print(f"  By Description: {change_events['Description'].value_counts().to_dict()}")
    print(f"  Unique sources: {change_events['Source'].nunique()}")


change_events already loaded by Section 3c  →  96,020 rows
Section 4 fallback skipped.


## Section 5: Trip Period Filtering

Removes plant shutdown / maintenance periods from both the PV/OP/SP time series and the events data.

In [7]:
if FILTER_TRIPS:
    trips_df = pd.read_csv(TRIP_FILE)
    trips_df['Stop Date']  = pd.to_datetime(trips_df['Stop Date'])
    trips_df['Start Date'] = pd.to_datetime(trips_df['Start Date'])
    print(f"Trip periods loaded : {len(trips_df)}")

    # ── Filter PV/OP/SP time series ──
    pre_pv       = len(op_pv_data_df)
    pv_trip_mask = pd.Series(False, index=op_pv_data_df.index)
    for _, trip in trips_df.iterrows():
        pv_trip_mask |= (
            (op_pv_data_df.index >= trip['Stop Date']) &
            (op_pv_data_df.index <= trip['Start Date'])
        )
    op_pv_data_df = op_pv_data_df[~pv_trip_mask]
    print(f"PV/OP/SP filter : {pre_pv:,} → {len(op_pv_data_df):,} rows "
          f"(removed {pre_pv - len(op_pv_data_df):,})")

    # ── Filter events ──
    if change_events is not None:
        pre_ev       = len(change_events)
        ev_trip_mask = pd.Series(False, index=change_events.index)
        for _, trip in trips_df.iterrows():
            ev_trip_mask |= (
                (change_events['VT_Start'] >= trip['Stop Date']) &
                (change_events['VT_Start'] <= trip['Start Date'])
            )
        change_events = change_events[~ev_trip_mask].reset_index(drop=True)
        print(f"Events filter   : {pre_ev:,} → {len(change_events):,} rows "
              f"(removed {pre_ev - len(change_events):,})")
    else:
        print("Events filter   : skipped (no events loaded)")
else:
    print("Trip filtering is OFF — skipped.")


Trip periods loaded : 106


PV/OP/SP filter : 2,102,317 → 2,002,613 rows (removed 99,704)
Events filter   : 96,020 → 76,479 rows (removed 19,541)


## Section 5b: Control Actions Stats — Tags & Volume per Period

Shows how many unique tags have OP/SP/MODE actions in each configured time period,  
and a ranked table of the most-operated tags across the full date range.

In [8]:
# ── Define period helper if Section 8 hasn't run yet ──
if '_get_periods' not in dir():
    _MONTH_ABBR = ['Jan','Feb','Mar','Apr','May','Jun',
                   'Jul','Aug','Sep','Oct','Nov','Dec']

    def _period_suffix(period_idx, period_months):
        if period_months == 6:
            return f'H{period_idx + 1}'
        if period_months == 3:
            return f'Q{period_idx + 1}'
        if period_months == 4:
            return f'T{period_idx + 1}'
        return f'P{period_idx + 1}'

    def _get_periods(years, period_months):
        periods_per_year = 12 // period_months
        result = []
        for year in sorted(years):
            for p in range(periods_per_year):
                start_month = p * period_months + 1
                end_month   = (p + 1) * period_months
                last_day    = calendar.monthrange(year, end_month)[1]
                w_start     = pd.Timestamp(year, start_month, 1)
                w_end       = pd.Timestamp(year, end_month, last_day, 23, 59, 59)
                suffix      = _period_suffix(p, period_months)
                month_range = f'{_MONTH_ABBR[start_month-1]}–{_MONTH_ABBR[end_month-1]}'
                label       = f'{year}-{suffix}  ({month_range})'
                filename    = f'period_{year}_{suffix}'
                result.append((filename, label, w_start, w_end))
        return result

if change_events is None:
    print("No events loaded — skipping stats.")
else:
    _periods = _get_periods(YEARS, PERIOD_MONTHS)

    print("╔══════════════════════════════════════════════════════════════════════╗")
    print("║           CONTROL ACTIONS STATS  (after trip filtering)            ║")
    print("╚══════════════════════════════════════════════════════════════════════╝")

    # ── Overall summary ──
    print(f"\n── Overall ({change_events['VT_Start'].min().date()} → "
          f"{change_events['VT_Start'].max().date()}) ──")
    print(f"  Total actions           : {len(change_events):,}")
    print(f"  By Description          : {change_events['Description'].value_counts().to_dict()}")
    print(f"  Unique tags with actions: {change_events['Source'].nunique()}")

    # Top 30 most operated tags (all time)
    tag_action_counts = (
        change_events.groupby('Source')
        .agg(
            total_actions =('VT_Start',    'count'),
            op_count      =('Description', lambda x: (x == 'OP').sum()),
            sp_count      =('Description', lambda x: (x == 'SP').sum()),
            mode_count    =('Description', lambda x: (x == 'MODE').sum()),
        )
        .sort_values('total_actions', ascending=False)
        .reset_index()
    )
    print(f"\nTop 30 most-operated tags (all time):")
    display(tag_action_counts.head(30))

    # ── Per-period breakdown ──
    print(f"\n── Per-period breakdown ──")
    period_rows = []
    for fn, lbl, ws, we in _periods:
        mask = (change_events['VT_Start'] >= ws) & (change_events['VT_Start'] <= we)
        ev_p = change_events.loc[mask]
        period_rows.append({
            'Period'           : lbl,
            'Total actions'    : len(ev_p),
            'OP'               : (ev_p['Description'] == 'OP').sum(),
            'SP'               : (ev_p['Description'] == 'SP').sum(),
            'MODE'             : (ev_p['Description'] == 'MODE').sum(),
            'Unique tags'      : ev_p['Source'].nunique(),
            'Tags with OP/SP'  : ev_p[ev_p['Description'].isin(['OP', 'SP'])]['Source'].nunique(),
        })
    df_period_stats = pd.DataFrame(period_rows)
    display(df_period_stats)

    # Per-period top-20 tags
    for fn, lbl, ws, we in _periods:
        mask = (change_events['VT_Start'] >= ws) & (change_events['VT_Start'] <= we)
        ev_p = change_events.loc[mask]
        if ev_p.empty:
            continue
        top_tags = (
            ev_p.groupby('Source')
            .agg(
                actions=('VT_Start',    'count'),
                op     =('Description', lambda x: (x == 'OP').sum()),
                sp     =('Description', lambda x: (x == 'SP').sum()),
                mode   =('Description', lambda x: (x == 'MODE').sum()),
            )
            .sort_values('actions', ascending=False)
            .head(20)
            .reset_index()
        )
        print(f"\n  {lbl} — top 20 tags by action count:")
        display(top_tags)


╔══════════════════════════════════════════════════════════════════════╗
║           CONTROL ACTIONS STATS  (after trip filtering)            ║
╚══════════════════════════════════════════════════════════════════════╝

── Overall (2024-01-01 → 2025-06-28) ──
  Total actions           : 76,479
  By Description          : {'OP': 51943, 'SP': 16132, 'MODE': 8404}
  Unique tags with actions: 322

Top 30 most-operated tags (all time):


,Source,total_actions,op_count,sp_count,mode_count
0,03FIC_3227,8261,4170,2891,1200
1,03LIC_1034,8067,314,7492,261
2,03LIC_1619,4769,3382,1017,370
3,03HIC_1720B,4207,3903,0,304
4,03PIC_1013,4106,3827,0,279
5,03FIC_3435,3211,2708,0,503
6,03TIC_1745A,2952,1902,18,1032
7,03FIC_1725,2925,2074,213,638
8,03HIC_3100,2275,1989,0,286
9,03HIC_1151,1727,1537,0,190



── Per-period breakdown ──


,Period,Total actions,OP,SP,MODE,Unique tags,Tags with OP/SP
0,2024-P1 (Jan–Jan),4124,2935,735,454,147,142
1,2024-P2 (Feb–Feb),3925,2956,518,451,147,141
2,2024-P3 (Mar–Mar),2970,1704,891,375,99,98
3,2024-P4 (Apr–Apr),4774,3313,897,564,167,160
4,2024-P5 (May–May),4243,2754,1084,405,142,139
5,2024-P6 (Jun–Jun),5944,4178,1180,586,132,129
6,2024-P7 (Jul–Jul),5810,4044,1194,572,191,188
7,2024-P8 (Aug–Aug),5751,4003,1208,540,97,94
8,2024-P9 (Sep–Sep),2937,1619,933,385,96,92
9,2024-P10 (Oct–Oct),3736,2185,1138,413,123,121



  2024-P1  (Jan–Jan) — top 20 tags by action count:


...

,Source,actions,op,sp,mode
0,03LIC_1034,439,38,385,16
1,03FIC_1725,221,187,0,34
2,03FIC_3227,168,89,30,49
3,03TIC_1745A,141,63,0,78
4,03FIC_3435,138,107,0,31
5,03HIC_1720B,132,120,0,12
6,03PIC_3002,83,83,0,0
7,03HIC_3100,70,65,0,5
8,03LIC_1619,69,48,15,6
9,03KM_0102,64,64,0,0



  2025-P5  (May–May) — top 20 tags by action count:


,Source,actions,op,sp,mode
0,03PIC_1013,892,844,0,48
1,03FIC_3435,770,709,0,61
2,03LIC_1094,643,563,39,41
3,03LIC_1034,544,21,497,26
4,03LIC_1619,478,373,47,58
5,03HIC_1151,391,371,0,20
6,03HIC_1720B,335,317,0,18
7,03HIC_2474A,169,157,0,12
8,03FIC_1725,168,119,0,49
9,03LIC_1085,164,0,162,2



  2025-P6  (Jun–Jun) — top 20 tags by action count:


,Source,actions,op,sp,mode
0,03LIC_1034,389,47,330,12
1,03PIC_1013,335,299,0,36
2,03FIC_3227,261,64,137,60
3,03FIC_3435,241,199,0,42
4,03HIC_1151,241,209,0,32
5,03FIC_1725,220,174,6,40
6,03TIC_1745A,104,39,0,65
7,03HIC_1720B,94,79,0,15
8,03PIC_1720,73,0,68,5
9,03LIC_1071,71,54,13,4


## Section 6: Colour Palette & Ordered Tag List

One consistent colour per base tag across all plots.  
Tag order per base: **PV → SP → OP** (SP uses the same colour with `dash='dashdot'`; OP uses `dash='dot'`).

In [27]:
# ── Collect all base tags present in the parquet (superset of TARGET_TAGS) ──
all_data_cols = [c for c in op_pv_data_df.columns
                 if c.endswith('.PV') or c.endswith('.OP') or c.endswith('.SP')]

pv_bases = {c.rsplit('.', 1)[0]: c for c in all_data_cols if c.endswith('.PV')}
op_bases = {c.rsplit('.', 1)[0]: c for c in all_data_cols if c.endswith('.OP')}
sp_bases = {c.rsplit('.', 1)[0]: c for c in all_data_cols if c.endswith('.SP')}
all_bases = sorted(set(list(pv_bases) + list(op_bases) + list(sp_bases)))

# ── Build ordered tag list: PV → SP → OP per base ──
ordered_tags = []
for base in all_bases:
    if base in pv_bases:
        ordered_tags.append(pv_bases[base])
    if base in sp_bases:
        ordered_tags.append(sp_bases[base])
    if base in op_bases:
        ordered_tags.append(op_bases[base])

# ── Assign a consistent colour per base tag ──
_palette = (
    px.colors.qualitative.Dark24
    + px.colors.qualitative.Light24
    + px.colors.qualitative.Alphabet
)
base_colors = {base: _palette[i % len(_palette)] for i, base in enumerate(all_bases)}

# ── Split tags into groups (one plot per group, capped at MAX_TAGS_PER_PLOT) ──
FAMILY_LABELS = {
    'controllers': 'Controllers', 'T': 'Temperature (TI)', 'F': 'Flow (FI)',
    'L': 'Level (LI)', 'P': 'Pressure (PI)', 'other': 'Other',
}
FAMILY_ORDER = ['controllers', 'T', 'F', 'L', 'P', 'other']

def base_family(base):
    """controllers (xIC/HIC/SIC) → 'controllers'; TI/FI/LI/PI → 'T'/'F'/'L'/'P'; rest → 'other'."""
    m = re.match(r'\d+([A-Z]+)_', base)
    itype = m.group(1) if m else ''
    if itype.endswith('C'):
        return 'controllers'
    if itype in {'TI', 'FI', 'LI', 'PI'}:
        return itype[0]
    return 'other'

# Bases per family (preserves all_bases order)
_fam_bases = {f: [b for b in all_bases if base_family(b) == f] for f in FAMILY_ORDER}

if SPLIT_TAGS_INTO_GROUPS:
    group_nav, group_ordered_tags, base_to_group = [], {}, {}
    for fam in FAMILY_ORDER:
        bases = _fam_bases[fam]
        if not bases:
            continue
        nchunks = -(-len(bases) // MAX_TAGS_PER_PLOT)   # ceil
        for ci in range(nchunks):
            chunk = bases[ci * MAX_TAGS_PER_PLOT:(ci + 1) * MAX_TAGS_PER_PLOT]
            key   = f'{fam}_{ci+1}' if nchunks > 1 else fam
            lbl   = FAMILY_LABELS[fam] + (f' {ci+1}/{nchunks}' if nchunks > 1 else '')
            cols  = [c for b in chunk for c in (pv_bases.get(b), sp_bases.get(b), op_bases.get(b)) if c]
            group_nav.append((key, f'{lbl} ({len(chunk)})'))
            group_ordered_tags[key] = cols
            for b in chunk:
                base_to_group[b] = key
else:
    group_nav = [('all', 'All tags')]
    group_ordered_tags = {'all': ordered_tags}
    base_to_group = {b: 'all' for b in all_bases}

print(f"Base tags in parquet : {len(all_bases)}")
print(f"Ordered traces       : {len(ordered_tags)}  "
      f"({len(pv_bases)} PV + {len(sp_bases)} SP + {len(op_bases)} OP)")
print(f"Colour palette       : {len(base_colors)} entries  |  cap {MAX_TAGS_PER_PLOT} tags/plot")
print(f"FI1000 column present: {FI1000_COL in op_pv_data_df.columns}")
print(f"\nTag groups ({len(group_nav)}):")
for key, _ in group_nav:
    cols = group_ordered_tags[key]
    nbase = len(set(c.rsplit('.', 1)[0] for c in cols))
    print(f"  {key:14s} : {nbase:3d} tags  |  {len(cols):3d} traces")


Base tags in parquet : 302
Ordered traces       : 361  (300 PV + 1 SP + 60 OP)
Colour palette       : 302 entries  |  cap 75 tags/plot
FI1000 column present: True

Tag groups (6):
  controllers    :  65 tags  |  124 traces
  T              :  65 tags  |   65 traces
  F              :  20 tags  |   20 traces
  L              :  20 tags  |   20 traces
  P              :  72 tags  |   72 traces
  other          :  60 tags  |   60 traces


## Section 7: Plot Helper Functions

- `create_period_plot()` — builds the normalized multi-tag trend figure with optional control-actions subplot
- `build_period_nav_html()` — builds the sticky navigation bar linking all period HTML files

In [34]:
def build_period_nav_html(periods, groups, current_period_stem, current_group):
    """Sticky nav bar with a tag-group selector + period selector. Full filename = f'{stem}_{group}'."""
    def fname(stem, gkey):
        return f'{stem}_{gkey}'

    p_options = [
        f'<option value="{fname(stem, current_group)}"'
        f'{" selected" if stem == current_period_stem else ""}>{lbl}</option>'
        for stem, lbl in periods
    ]
    g_options = [
        f'<option value="{fname(current_period_stem, gk)}"'
        f'{" selected" if gk == current_group else ""}>{glbl}</option>'
        for gk, glbl in groups
    ]

    cur_idx    = next(i for i, (stem, _) in enumerate(periods) if stem == current_period_stem)
    prev_entry = periods[cur_idx - 1] if cur_idx > 0 else None
    next_entry = periods[cur_idx + 1] if cur_idx < len(periods) - 1 else None

    def btn(label, stem, disabled=False):
        if disabled:
            return (f'<button disabled style="padding:6px 14px;border:1px solid #eee;'
                    f'border-radius:4px;background:#f0f0f0;color:#aaa;">{label}</button>')
        return (f'<button onclick="window.location.href=\'{fname(stem, current_group)}.html\'" '
                f'style="padding:6px 14px;cursor:pointer;border:1px solid #ccc;'
                f'border-radius:4px;background:#f8f8f8;">{label}</button>')

    prev_btn = btn(f'◀ {prev_entry[1]}', prev_entry[0]) if prev_entry else btn('◀ Prev', '', disabled=True)
    next_btn = btn(f'{next_entry[1]} ▶', next_entry[0]) if next_entry else btn('Next ▶', '', disabled=True)

    return f'''
    <div style="position:sticky;top:0;z-index:9999;background:#fff;padding:10px 15px;
                border-bottom:2px solid #ddd;display:flex;align-items:center;gap:12px;
                font-family:Arial,sans-serif;font-size:14px;">
        <span style="font-weight:bold;color:#333;">Group:</span>
        <select onchange="window.location.href=this.value+'.html'"
                style="padding:6px 10px;border:1px solid #1976D2;border-radius:4px;
                       font-size:13px;background:#e3f2fd;font-weight:bold;min-width:230px;">
            {"".join(g_options)}
        </select>
        <span style="font-weight:bold;color:#333;margin-left:8px;">Period:</span>
        {prev_btn}
        <select onchange="window.location.href=this.value+'.html'"
                style="padding:6px 10px;border:1px solid #ccc;border-radius:4px;
                       font-size:13px;min-width:230px;">
            {"".join(p_options)}
        </select>
        {next_btn}
        <span style="color:#888;font-size:12px;margin-left:auto;">
            {cur_idx + 1} of {len(periods)} periods
        </span>
    </div>
    '''


def build_tag_filter_html(sources, top_10_tags, group_key='all'):
    """Tag filter panel for the actions y-axis. Per-group storage; default = show ALL tags."""
    all_tags_js  = json.dumps(sources)
    top10_js     = json.dumps(top_10_tags)
    group_key_js = json.dumps(group_key)

    return f'''
    <div id="tag-filter-panel" style="background:#f9f9f9;border:1px solid #ddd;border-radius:6px;
                padding:12px 16px;margin:10px 0;font-family:Arial,sans-serif;font-size:13px;">
        <div style="display:flex;align-items:center;gap:10px;margin-bottom:8px;">
            <span style="font-weight:bold;color:#333;">Filter Actions Y-Axis:</span>
            <input id="tag-search-input" type="text" placeholder="Type to search tags..."
                   style="padding:6px 10px;border:1px solid #ccc;border-radius:4px;
                          font-size:13px;width:220px;" oninput="filterTags()">
            <button onclick="showAllTags()" style="padding:5px 12px;cursor:pointer;border:1px solid #4CAF50;
                    border-radius:4px;background:#e8f5e9;color:#2e7d32;font-size:12px;font-weight:bold;">
                All ({len(sources)})</button>
            <button onclick="showTopTags()" style="padding:5px 12px;cursor:pointer;border:1px solid #1976D2;
                    border-radius:4px;background:#e3f2fd;color:#1565C0;font-size:12px;font-weight:bold;">
                Top 10</button>
            <button onclick="clearAllTags()" style="padding:5px 12px;cursor:pointer;border:1px solid #e57373;
                    border-radius:4px;background:#ffebee;color:#c62828;font-size:12px;font-weight:bold;">
                None</button>
            <span id="filter-status" style="color:#888;font-size:11px;margin-left:auto;"></span>
        </div>
        <div id="tag-chips" style="display:flex;flex-wrap:wrap;gap:4px;max-height:120px;
                    overflow-y:auto;padding:4px 0;"></div>
    </div>

    <script>
    (function() {{
        const ALL_TAGS    = {all_tags_js};
        const TOP10       = {top10_js};
        // Single shared key → selection persists across ALL groups AND months.
        const STORAGE_KEY = 'rca_action_sel_shared';
        const plotDiv   = document.querySelector('.plotly-graph-div');
        const chipsDiv  = document.getElementById('tag-chips');
        const statusEl  = document.getElementById('filter-status');
        const searchBox = document.getElementById('tag-search-input');

        function loadSavedSet() {{
            try {{ const raw = localStorage.getItem(STORAGE_KEY); if (raw) return new Set(JSON.parse(raw)); }}
            catch (e) {{}}
            return null;
        }}
        function saveSelection() {{
            try {{
                // Merge-preserve: keep selections for tags not in this plot, overwrite this plot's tags.
                const saved = loadSavedSet() || new Set();
                ALL_TAGS.forEach(t => saved.delete(t));
                selectedTags.forEach(t => saved.add(t));
                localStorage.setItem(STORAGE_KEY, JSON.stringify(Array.from(saved)));
            }} catch (e) {{}}
        }}

        const _savedSet = loadSavedSet();
        let selectedTags = _savedSet
            ? new Set(ALL_TAGS.filter(t => _savedSet.has(t)))
            : new Set(TOP10);

        function updateStatus() {{
            const shown = ALL_TAGS.filter(t => selectedTags.has(t)).length;
            statusEl.textContent = shown + ' of ' + ALL_TAGS.length + ' tags shown';
        }}
        function renderChips(filter) {{
            chipsDiv.innerHTML = '';
            const lower = (filter || '').toLowerCase();
            ALL_TAGS.filter(t => !lower || t.toLowerCase().includes(lower)).forEach(tag => {{
                const chip = document.createElement('span');
                chip.textContent = tag;
                chip.style.cssText = 'padding:3px 8px;border-radius:3px;cursor:pointer;font-size:11px;' +
                    (selectedTags.has(tag) ? 'background:#1976D2;color:#fff;border:1px solid #1565C0;'
                                           : 'background:#fff;color:#555;border:1px solid #ccc;');
                chip.onclick = () => toggleTag(tag);
                chipsDiv.appendChild(chip);
            }});
            updateStatus();
        }}
        function toggleTag(tag) {{
            if (selectedTags.has(tag)) selectedTags.delete(tag); else selectedTags.add(tag);
            applyFilter(); saveSelection(); renderChips(searchBox.value);
        }}
        function applyFilter() {{
            if (!plotDiv) return;
            const selected   = ALL_TAGS.filter(t => selectedTags.has(t));
            const unselected = ALL_TAGS.filter(t => !selectedTags.has(t));
            const reordered  = selected.concat(unselected);
            const n = selected.length;
            // n>0 → show first n categories; n==0 → push range below 0 so none show
            Plotly.relayout(plotDiv, {{
                'yaxis2.categoryarray': reordered,
                'yaxis2.categoryorder': 'array',
                'yaxis2.range': n > 0 ? [-0.5, n - 0.5] : [-1.0, -0.5]
            }});
            updateStatus();
        }}
        window.filterTags  = function() {{
            const q = searchBox.value.toLowerCase();
            if (q.length >= 2) {{ selectedTags = new Set(ALL_TAGS.filter(t => t.toLowerCase().includes(q)));
                                  applyFilter(); saveSelection(); }}
            renderChips(q);
        }};
        window.showAllTags = function() {{ searchBox.value=''; selectedTags=new Set(ALL_TAGS); applyFilter(); saveSelection(); renderChips(''); }};
        window.showTopTags = function() {{ searchBox.value=''; selectedTags=new Set(TOP10);    applyFilter(); saveSelection(); renderChips(''); }};
        window.clearAllTags= function() {{ searchBox.value=''; selectedTags=new Set();         applyFilter(); saveSelection(); renderChips(''); }};

        renderChips(''); applyFilter();
    }})();
    </script>
    '''


def create_period_plot(window_start, window_end, op_pv_df, ordered_tags, base_colors,
                       title='', actions=None, operating_limits=None, fi1000_col=None):
    """Normalized multi-tag PV/SP/OP trend plot for a window, with optional control-actions subplot.
       Returns (fig, sources_list) or (None, None)."""
    mask      = (op_pv_df.index >= window_start) & (op_pv_df.index <= window_end)
    window_df = op_pv_df.loc[mask, [c for c in ordered_tags if c in op_pv_df.columns]].copy()

    if window_df.empty:
        print(f"  No PV/OP data in window {window_start} → {window_end}")
        return None, None

    # Downsample for plotting only (raw 1-min stays in CSV) — main speed/size lever
    if PLOT_RESAMPLE:
        window_df = window_df.resample(PLOT_RESAMPLE).mean()

    has_actions = actions is not None and len(actions) > 0

    if has_actions:
        n_act_tags  = actions['Source'].nunique()
        act_height  = max(0.15, min(0.40, n_act_tags * 0.025))
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                            row_heights=[1 - act_height, act_height],
                            subplot_titles=('PV / SP / OP Trends (min–max normalised)',
                                            'Control Actions  (OP / SP / MODE)'))
    else:
        fig = make_subplots(rows=1, cols=1,
                            subplot_titles=('PV / SP / OP Trends (min–max normalised)',))

    x_range = [window_df.index[0], window_df.index[-1]]
    LIMIT_CLAMP_LO, LIMIT_CLAMP_HI = -0.04, 1.04
    YAXIS_RANGE_LO, YAXIS_RANGE_HI = -0.10, 1.10

    # ── Trend traces ──
    for col in ordered_tags:
        if col not in window_df.columns:
            continue
        series = window_df[col]
        if series.isna().all():
            continue
        col_min, col_max = series.min(), series.max()
        normalized = (pd.Series(0.5, index=series.index) if col_max == col_min
                      else (series - col_min) / (col_max - col_min))
        parts  = col.rsplit('.', 1)
        base   = parts[0]
        suffix = parts[1] if len(parts) == 2 else ''
        color  = base_colors.get(base, '#888888')
        dash   = {'OP': 'dot', 'SP': 'dashdot'}.get(suffix, 'solid')
        width  = 1.8 if suffix == 'PV' else 1.4
        visible = True if (fi1000_col and col == fi1000_col) else 'legendonly'

        fig.add_trace(go.Scatter(
            x=series.index, y=normalized, mode='lines', name=col,
            legendgroup=base, legendgrouptitle_text=base if suffix == 'PV' else None,
            line=dict(color=color, width=width, dash=dash), visible=visible,
            customdata=np.column_stack([series.values]),
            hovertemplate=(f'<b>{col}</b><br>Time: %{{x}}<br>Value: %{{customdata[0]:.4f}}<extra></extra>'),
        ), row=1, col=1)

        if (operating_limits and suffix == 'PV' and col in operating_limits and col_max != col_min):
            lims  = operating_limits[col]
            n_up_c = min(max((lims['upper'] - col_min) / (col_max - col_min), LIMIT_CLAMP_LO), LIMIT_CLAMP_HI)
            n_lo_c = min(max((lims['lower'] - col_min) / (col_max - col_min), LIMIT_CLAMP_LO), LIMIT_CLAMP_HI)
            fig.add_trace(go.Scatter(
                x=[x_range[0], x_range[1], None, x_range[0], x_range[1]],
                y=[n_up_c, n_up_c, None, n_lo_c, n_lo_c], mode='lines+text',
                text=[f'{base} UL={lims["upper"]:.2f}', '', '', f'{base} LL={lims["lower"]:.2f}', ''],
                textposition=['bottom right', 'bottom right', 'bottom right', 'top right', 'top right'],
                textfont=dict(color=color, size=9),
                name=f'{col} limits ({lims["lower"]:.2f}–{lims["upper"]:.2f})',
                legendgroup=base, line=dict(color=color, width=1.2, dash='dash'),
                visible='legendonly', showlegend=True, cliponaxis=False,
                hovertemplate=(f'<b>{col} limits: {lims["lower"]:.2f} – {lims["upper"]:.2f}</b><extra></extra>'),
            ), row=1, col=1)

    # ── Control actions subplot — grouped by action type + direction ──
    sources = []
    if has_actions:
        sources = sorted(actions['Source'].unique())
        action_groups = [
            ('OP ↑', 'green', 'circle',  lambda df: df[(df['Description'] == 'OP') & (df['action_direction'] == 'increase')]),
            ('OP ↓', 'red',   'circle',  lambda df: df[(df['Description'] == 'OP') & (df['action_direction'] == 'decrease')]),
            ('SP ↑', 'green', 'diamond', lambda df: df[(df['Description'] == 'SP') & (df['action_direction'] == 'increase')]),
            ('SP ↓', 'red',   'diamond', lambda df: df[(df['Description'] == 'SP') & (df['action_direction'] == 'decrease')]),
            ('MODE', 'blue',  'square',  lambda df: df[df['Description'] == 'MODE']),
        ]
        for grp_label, grp_color, grp_symbol, grp_filter in action_groups:
            grp_df = grp_filter(actions)
            if grp_df.empty:
                continue
            hover_texts = [
                f'<b>{row["Source"]}</b> ({row["Description"]})<br>Time: {row["VT_Start"]}<br>'
                f'{row.get("PrevValue","?")} → {row.get("Value","?")}<br>'
                f'Direction: {row.get("action_direction","")}<extra></extra>'
                for _, row in grp_df.iterrows()
            ]
            fig.add_trace(go.Scatter(
                x=grp_df['VT_Start'].values, y=grp_df['Source'].values, mode='markers',
                marker=dict(color=grp_color, size=8, symbol=grp_symbol, line=dict(width=1, color='darkgrey')),
                name=grp_label, legendgroup='_ctrl_actions', legendgrouptitle_text='Control Actions',
                visible=True, hovertemplate=hover_texts,
            ), row=2, col=1)

        top_10_default = actions['Source'].value_counts().head(10).index.tolist()
        other_tags = [t for t in sources if t not in top_10_default]
        fig.update_yaxes(type='category', categoryorder='array',
                         categoryarray=top_10_default + other_tags,
                         range=[-0.5, len(top_10_default) - 0.5], row=2, col=1,
                         title_text='', gridcolor='rgba(200,200,200,0.3)', tickfont=dict(size=9))

    fig.update_layout(title=title, height=850 if has_actions else 620, template='plotly_white',
                      hovermode='closest',
                      legend=dict(groupclick='toggleitem', tracegroupgap=3, font=dict(size=10)))
    fig.update_yaxes(showticklabels=False, title_text='',
                     range=[YAXIS_RANGE_LO, YAXIS_RANGE_HI], autorange=False, row=1, col=1)
    return fig, sources


print("create_period_plot(), build_period_nav_html(), and build_tag_filter_html() defined.")


create_period_plot(), build_period_nav_html(), and build_tag_filter_html() defined.


## Section 8: Generate Period Plots

For each year × period window:
- Filters the time series and events to the window
- Generates **one normalised plot per tag group** (capped at `MAX_TAGS_PER_PLOT`) so no single file is overloaded
- Exports a PV/OP/SP CSV and (if events loaded) an events CSV per period
- Injects a navigation bar with **group** + **period** selectors to browse in the browser

**Switch from 6-month to quarterly** by setting `PERIOD_MONTHS = 3` in Section 1 and re-running from Section 8.

In [29]:
_MONTH_ABBR = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']


def _period_suffix(period_idx, period_months):
    """Return a short period suffix label: H1/H2, Q1–Q4, T1–T3, or P1–Pn."""
    if period_months == 6:
        return f'H{period_idx + 1}'
    if period_months == 3:
        return f'Q{period_idx + 1}'
    if period_months == 4:
        return f'T{period_idx + 1}'
    return f'P{period_idx + 1}'


def _get_periods(years, period_months):
    """
    Return a list of (filename_stem, label, window_start, window_end) for all
    year × period combinations, in chronological order.
    """
    periods_per_year = 12 // period_months
    result = []
    for year in sorted(years):
        for p in range(periods_per_year):
            start_month = p * period_months + 1
            end_month   = (p + 1) * period_months
            last_day    = calendar.monthrange(year, end_month)[1]
            w_start     = pd.Timestamp(year, start_month, 1)
            w_end       = pd.Timestamp(year, end_month, last_day, 23, 59, 59)
            suffix      = _period_suffix(p, period_months)
            month_range = f'{_MONTH_ABBR[start_month-1]}–{_MONTH_ABBR[end_month-1]}'
            label       = f'{year}-{suffix}  ({month_range})'
            filename    = f'period_{year}_{suffix}'
            result.append((filename, label, w_start, w_end))
    return result


# ── Build the full period list (used for the navigation bar in every file) ──
all_periods_meta = _get_periods(YEARS, PERIOD_MONTHS)
# all_periods (without window bounds) fed to build_period_nav_html
all_periods_nav  = [(fn, lbl) for fn, lbl, _, _ in all_periods_meta]
# group_nav is built in Section 6 (capped subgroups). Fallback for a stale kernel:
if 'group_nav' not in dir():
    group_nav = [('all', 'All tags')]

print(f"Periods to generate : {len(all_periods_meta)}")
for fn, lbl, ws, we in all_periods_meta:
    print(f"  {fn:20s}  {lbl}  |  {ws.date()} → {we.date()}")
print(f"\nGroups per period   : {len(group_nav)}  →  {[g[0] for g in group_nav]}")
print(f"Total HTML files    : {len(all_periods_meta) * len(group_nav)}")
print(f"Output directory : {RESULTS_DIR}")


Periods to generate : 24
  period_2024_P1        2024-P1  (Jan–Jan)  |  2024-01-01 → 2024-01-31
  period_2024_P2        2024-P2  (Feb–Feb)  |  2024-02-01 → 2024-02-29
  period_2024_P3        2024-P3  (Mar–Mar)  |  2024-03-01 → 2024-03-31
  period_2024_P4        2024-P4  (Apr–Apr)  |  2024-04-01 → 2024-04-30
  period_2024_P5        2024-P5  (May–May)  |  2024-05-01 → 2024-05-31
  period_2024_P6        2024-P6  (Jun–Jun)  |  2024-06-01 → 2024-06-30
  period_2024_P7        2024-P7  (Jul–Jul)  |  2024-07-01 → 2024-07-31
  period_2024_P8        2024-P8  (Aug–Aug)  |  2024-08-01 → 2024-08-31
  period_2024_P9        2024-P9  (Sep–Sep)  |  2024-09-01 → 2024-09-30
  period_2024_P10       2024-P10  (Oct–Oct)  |  2024-10-01 → 2024-10-31
  period_2024_P11       2024-P11  (Nov–Nov)  |  2024-11-01 → 2024-11-30
  period_2024_P12       2024-P12  (Dec–Dec)  |  2024-12-01 → 2024-12-31
  period_2025_P1        2025-P1  (Jan–Jan)  |  2025-01-01 → 2025-01-31
  period_2025_P2        2025-P2  (Feb–Feb)  |  20

In [35]:
for filename, label, w_start, w_end in all_periods_meta:

    print(f"\n{'─'*60}")
    print(f"  {label}  ({w_start.date()} → {w_end.date()})")

    # ── Slice PV/OP/SP data ──
    pv_mask   = (op_pv_data_df.index >= w_start) & (op_pv_data_df.index <= w_end)
    pv_window = op_pv_data_df.loc[pv_mask].copy()
    print(f"  PV/OP/SP rows in window : {len(pv_window):,}")

    if pv_window.empty:
        print("  Skipping — no time-series data in this window.")
        continue

    # ── Slice events ──
    if change_events is not None:
        ev_mask   = (change_events['VT_Start'] >= w_start) & (change_events['VT_Start'] <= w_end)
        ev_window = change_events.loc[ev_mask].copy().reset_index(drop=True)
        print(f"  Actions in window       : {len(ev_window):,}  "
              f"{ev_window['Description'].value_counts().to_dict()}")
    else:
        ev_window = None

    # ── One plot per tag group ──
    for gkey, glabel in group_nav:
        g_cols = [c for c in group_ordered_tags.get(gkey, []) if c in pv_window.columns]
        if not g_cols:
            continue

        # Show the FULL control-actions timeline on every group plot (not just
        # the group's own tags) so trends can be correlated with all operator moves.
        g_actions = ev_window if (ev_window is not None and not ev_window.empty) else None
        n_actions = 0 if g_actions is None else len(g_actions)

        n_traces = sum(1 for c in g_cols if not pv_window[c].isna().all())
        title = (
            f'{glabel}  |  {label}  |  {n_traces} traces'
            + (f'  |  {n_actions:,} control actions' if n_actions > 0 else '  |  no events')
        )

        fig, action_sources = create_period_plot(
            w_start, w_end,
            op_pv_df=pv_window,
            ordered_tags=g_cols,
            base_colors=base_colors,
            title=title,
            actions=g_actions,
            operating_limits=tag_operating_limits,
            fi1000_col=FI1000_COL,
        )
        if fig is None:
            print(f"    {gkey:12s}: skipped (no data).")
            continue

        # ── Inject nav + tag filter, save HTML ──
        nav_html  = build_period_nav_html(all_periods_nav, group_nav, filename, gkey)
        plotlyjs  = True if PLOTLY_EMBED_JS else 'cdn'
        plot_html = fig.to_html(include_plotlyjs=plotlyjs, full_html=True)
        plot_html = plot_html.replace('<body>', f'<body>\n{nav_html}', 1)

        if action_sources:
            top_10 = (g_actions['Source'].value_counts().head(10).index.tolist()
                      if g_actions is not None else [])
            filter_html = build_tag_filter_html(action_sources, top_10, gkey)
            plot_html = plot_html.replace('</body>', f'{filter_html}\n</body>', 1)

        html_path = RESULTS_DIR / f'{filename}_{gkey}.html'
        with open(html_path, 'w') as f:
            f.write(plot_html)
        print(f"    {gkey:12s}: {n_traces} traces, {n_actions:,} actions  → {html_path.name}")

    # ── Export PV/OP/SP CSV (full window, once per period) ──
    pv_csv = RESULTS_DIR / f'{filename}_pv_data.csv'
    pv_window.to_csv(pv_csv)
    print(f"  PV CSV      → {pv_csv.name}  ({len(pv_window):,} rows)")

    # ── Export events CSV ──
    if ev_window is not None and not ev_window.empty:
        ev_csv = RESULTS_DIR / f'{filename}_events.csv'
        ev_window.to_csv(ev_csv, index=False)
        print(f"  Events CSV  → {ev_csv.name}  ({len(ev_window):,} rows)")

print(f"\n{'═'*60}")
print(f"Done.  {len(all_periods_meta)} period(s) × {len(group_nav)} group(s) processed.")
print(f"Output : {RESULTS_DIR}")


────────────────────────────────────────────────────────────
  2024-P1  (Jan–Jan)  (2024-01-01 → 2024-01-31)


  PV/OP/SP rows in window : 43,967
  Actions in window       : 4,124  {'OP': 2935, 'SP': 735, 'MODE': 454}
    controllers : 121 traces, 4,124 actions  → period_2024_P1_controllers.html
    T           : 65 traces, 4,124 actions  → period_2024_P1_T.html
    F           : 20 traces, 4,124 actions  → period_2024_P1_F.html
    L           : 20 traces, 4,124 actions  → period_2024_P1_L.html
    P           : 72 traces, 4,124 actions  → period_2024_P1_P.html
    other       : 60 traces, 4,124 actions  → period_2024_P1_other.html
  PV CSV      → period_2024_P1_pv_data.csv  (43,967 rows)
  Events CSV  → period_2024_P1_events.csv  (4,124 rows)

────────────────────────────────────────────────────────────
  2024-P2  (Feb–Feb)  (2024-02-01 → 2024-02-29)
  PV/OP/SP rows in window : 41,243
  Actions in window       : 3,925  {'OP': 2956, 'SP': 518, 'MODE': 451}
    controllers : 121 traces, 3,925 actions  → period_2024_P2_controllers.html
    T           : 65 traces, 3,925 actions  → period_2024_P2

In [36]:
# ═══════════════════════════════════════════════════════════════════════════════
# Tag → Group Guide  (so anyone can tell which tags live in which HTML file)
# Writes 3 artefacts into RESULTS_DIR:
#   tag_group_guide.csv  — one row per tag: base, instrument type, group, PV/SP/OP, limits
#   README.md            — short text legend (file-naming pattern + per-group tag list)
#   index.html           — clickable landing page: period × group links + tag chips
# ═══════════════════════════════════════════════════════════════════════════════
import re

# Per-group description derived from family label
group_labels = dict(group_nav)
GROUP_DESC = {gk: glbl for gk, glbl in group_nav}

# ── 1. Build the tag→group table ──
guide_rows = []
for base in all_bases:
    m = re.match(r'\d+([A-Z]+)_', base)
    grp = base_to_group.get(base, 'other')
    lims = tag_operating_limits.get(f'{base}.PV', {})
    guide_rows.append({
        'tag'        : base,
        'instrument' : m.group(1) if m else '',
        'group'      : grp,
        'PV'         : 'Y' if f'{base}.PV' in op_pv_data_df.columns else '',
        'SP'         : 'Y' if f'{base}.SP' in op_pv_data_df.columns else '',
        'OP'         : 'Y' if f'{base}.OP' in op_pv_data_df.columns else '',
        'lower_limit': lims.get('lower', ''),
        'upper_limit': lims.get('upper', ''),
    })
df_guide = pd.DataFrame(guide_rows).sort_values(['group', 'instrument', 'tag']).reset_index(drop=True)
df_guide.to_csv(RESULTS_DIR / 'tag_group_guide.csv', index=False)

group_tags = {gk: df_guide.loc[df_guide['group'] == gk, 'tag'].tolist() for gk, _ in group_nav}

# ── 2. README.md ──
md = ['# Tag-Group Plot Guide', '',
      f'Generated: {pd.Timestamp.now(tz="Asia/Kolkata"):%d %b %Y %H:%M IST}',
      f'Period length: {PERIOD_MONTHS} month(s)  |  Years: {YEARS}', '',
      '## File naming', '`period_<YEAR>_P<MONTH>_<group>.html`  e.g. `period_2024_P1_controllers.html`',
      'Each month → one file per group. Open any file and use the **Group / Period** dropdowns to navigate.', '',
      '## Groups']
for gk, glbl in group_nav:
    md += [f'### {glbl}  (`{gk}`) — {len(group_tags[gk])} tags',
           GROUP_DESC.get(gk, ''), '', '`' + '`, `'.join(group_tags[gk]) + '`', '']
(RESULTS_DIR / 'README.md').write_text('\n'.join(md))

# ── 3. index.html landing page ──
cards = []
for gk, glbl in group_nav:
    chips = ''.join(f'<span style="display:inline-block;padding:2px 7px;margin:2px;'
                    f'background:#eef;border:1px solid #ccd;border-radius:3px;font-size:11px;">{t}</span>'
                    for t in group_tags[gk])
    cards.append(f'<div style="border:1px solid #ddd;border-radius:6px;padding:12px;margin:8px 0;">'
                 f'<h3 style="margin:0 0 4px;">{glbl} <span style="color:#888;font-weight:normal;">'
                 f'({len(group_tags[gk])} tags)</span></h3>'
                 f'<p style="color:#555;margin:0 0 6px;font-size:13px;">{GROUP_DESC.get(gk,"")}</p>{chips}</div>')

rows = ''.join(
    '<tr><td style="padding:4px 10px;">' + lbl + '</td>' +
    ''.join(f'<td style="padding:4px 8px;"><a href="{fn}_{gk}.html">{gk}</a></td>' for gk, _ in group_nav) +
    '</tr>' for fn, lbl, _, _ in all_periods_meta)

(RESULTS_DIR / 'index.html').write_text(
    f'<html><head><meta charset="utf-8"><title>Tag-Group Plots</title>'
    f'<style>body{{font-family:Arial;margin:24px;}}td,th{{border-bottom:1px solid #eee;text-align:left;}}</style></head>'
    f'<body><h1>Tag-Group Trend Plots</h1>'
    f'<p>Period: {PERIOD_MONTHS} month(s) · {YEARS}. Each month has 3 group files; use the in-plot dropdowns too.</p>'
    f'<h2>Groups</h2>{"".join(cards)}'
    f'<h2>Open a plot</h2><table><tr><th>Period</th>' +
    ''.join(f'<th>{gk}</th>' for gk, _ in group_nav) + f'</tr>{rows}</table></body></html>')

print('Guide written to', RESULTS_DIR)
print('  tag_group_guide.csv  README.md  index.html')
display(df_guide.groupby('group')['tag'].count().rename('tags'))


Guide written to /home/h604827/ControlActions/RESULTS/new_rca_plots_29JUN2026_1337
  tag_group_guide.csv  README.md  index.html


group
F              20
L              20
P              72
T              65
controllers    65
other          60
Name: tags, dtype: int64